In [ ]:
from pathlib import Path
import numpy as np
# from diffusion_policy_gml import utils
import matplotlib.pyplot as plt
import load_gml

In [ ]:
root = Path('runs')
# for run in root.iterdir():
#     print(run)

In [ ]:
folder = root / 'Jul05_14-28-29_eagle'
with np.load(folder / 'output_example.npz') as data:
    x_out = data['x_out']
    x_train = data['x_train']
print(x_out.shape, np.nanmin(x_out), np.nanmax(x_out))
print(x_train.shape, np.min(x_train), np.max(x_train))

In [ ]:
for stroke in x_out:
    plt.plot(*stroke[:, :2].T);
plt.axis('equal')

In [ ]:
SCREEN_DIM = 1000
def stroke2txy(stroke, t0=0):
    # total number of points is when the first nan appears
    T = np.where(np.isnan(stroke[:, 0]))[0]
    T = len(stroke) if len(T) == 0 else T[0]
    t = np.linspace(0, 2, T) + t0
    xy = stroke[:T, :2]
    xy = (xy + 3) / 6 * SCREEN_DIM
    return np.concatenate([t[:, None], xy], axis=1)

def strokes2txys(strokes):
    ret = []
    for stroke in strokes:
        ret.append(stroke2txy(stroke, t0=ret[-1][-1, 0] if ret else 0))
    return ret

# txy = stroke2txy(x_out[0])
txys = strokes2txys(x_out)
js = load_gml.txy_to_gml_json(txys, [0, SCREEN_DIM, 0, SCREEN_DIM])
with open(folder / 'output_example.json', 'w') as f:
    f.write(js)
js = load_gml.txy_to_gml_json(txys, [0, SCREEN_DIM, 0, SCREEN_DIM], alt_format=True)
with open(folder / 'output_example.js', 'w') as f:
    f.write(f'load_gml({js})')
print(folder / 'output_example.json')